# 归档材料

选中的轨迹来自他车预测；下方 rollout 不消费它。这是保留供辨析的历史实现，学习闭环请使用新的起步单元。

[归档索引](../README.md) · [当前学习入口](../../../course/first_loop/README.md)

# 07 · Planning & Closed Loop：预测怎样改变驾驶行为？

本章合并旧版 `00E` 的 planning/control 入口和 `09`。Prediction 输出的是可能的 agent futures；planner 需要在 route、碰撞风险、舒适性和车辆可行性之间做决策；controller 再把 trajectory 变成动作。最关键的转折是：**动作会改变下一帧输入**，因此 open-loop 好看不代表 closed-loop 安全。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import matplotlib.pyplot as plt

prediction = load_numpy_artifact("06_prediction.npz")
candidate = prediction["candidates"]
probabilities = prediction["probabilities"]
horizon = prediction["horizon_s"]

def choose_plan(candidates, probabilities, risk_weight=1.0):
    # Cut-in mode is the second mode in the shared artifact.
    progress = -candidates[:, -1, 1]
    risk_penalty = np.array([0.0, 3.0, 0.7]) * risk_weight
    cost = -progress + risk_penalty - np.log(probabilities + 1e-6)
    return int(np.argmin(cost)), cost

selected_mode, costs = choose_plan(candidate, probabilities, risk_weight=1.0)
selected = candidate[selected_mode]
print("selected mode:", selected_mode, "costs:", costs.round(3))

In [ ]:
def rollout(policy="guarded", dt=0.1, horizon_s=6.0, seed=4):
    rng = np.random.default_rng(seed)
    steps = int(horizon_s / dt)
    ego_x, ego_speed = 0.0, 8.0
    obstacle_x, obstacle_y = 18.0, 0.0
    rows = []
    for step in range(steps):
        time = step * dt
        obstacle_speed = 4.5 if time < 2.0 else 2.0
        gap = obstacle_x - ego_x
        lateral_risk = abs(obstacle_y) < 1.3
        risk = gap < 12.0 and lateral_risk
        if policy == "naive":
            target_speed = 8.0
            action = "nominal"
        elif risk:
            target_speed = max(0.0, obstacle_speed - 0.5)
            action = "degraded" if gap > 6.0 else "minimal_risk"
        else:
            target_speed = 8.0
            action = "learned_plan"
        acceleration = np.clip((target_speed - ego_speed) * 1.5, -4.0, 2.0)
        ego_speed = max(0.0, ego_speed + acceleration * dt)
        ego_x += ego_speed * dt
        obstacle_x += obstacle_speed * dt
        obstacle_y += rng.normal(0.0, 0.01)
        rows.append({"time_s": time, "gap_m": obstacle_x - ego_x, "ego_speed": ego_speed,
                     "acceleration": acceleration, "action": action})
    return rows

traces = {policy: rollout(policy) for policy in ["naive", "guarded"]}
for policy, rows in traces.items():
    frame = np.array([[row["gap_m"], row["ego_speed"], row["acceleration"]] for row in rows])
    jerk = np.diff(frame[:, 2], prepend=frame[0, 2]) / 0.1
    print(policy, {"min_gap_m": round(float(frame[:, 0].min()), 3),
                    "progress_m": round(float(frame[:, 1].sum() * 0.1), 3),
                    "max_jerk": round(float(np.abs(jerk).max()), 3),
                    "fallback_s": round(sum(r["action"] != "learned_plan" for r in rows) * 0.1, 2)})

for policy, rows in traces.items():
    plt.plot([r["time_s"] for r in rows], [r["gap_m"] for r in rows], label=policy)
plt.axhline(2.0, color="red", linestyle="--", label="collision threshold")
plt.legend()
plt.xlabel("time / s")
plt.ylabel("gap / m")
plt.title("Actions change the next observation")
plt.show()

## Planner/control contract and exercises

- planner: trajectory/maneuver with time, frame, feasibility and confidence;
- controller: current ego state + trajectory → steering/throttle/brake;
- safety layer: independent constraints and fallback, not a second name for the planner.

练习：改变 reaction time、observation noise 和 risk weight；比较 open-loop trajectory error 与 closed-loop minimum gap；说明为什么一个 planner 只优化 geometric distance 可能把 cut-in 直接撞上。

In [ ]:
risk_weights = np.linspace(0.0, 3.0, 13)
chosen = [choose_plan(candidate, probabilities, weight)[0] for weight in risk_weights]
print("risk weight → selected mode:", list(zip(risk_weights.round(2), chosen)))
save_numpy_artifact("07_planner.npz", selected_mode=np.array([selected_mode]), selected_trajectory=selected,
                    risk_weights=risk_weights, selected_modes=np.asarray(chosen))
save_json_artifact("07_planner_metrics.json", {
    "selected_mode": int(selected_mode), "mode_names": ["keep_lane", "cut_in", "brake"],
    "closed_loop_policies": ["naive", "guarded"], "next": "08_data_and_evaluation.ipynb",
})
print("saved planner artifact")

### 完成标准

你应能解释 prediction→planning→control 的边界，以及为什么 closed-loop 会改变评估分布。下一章会把同一条链放进 scenario bundle、log replay、metrics 和 corner-case slice，而不是继续增加孤立 demo。